## Introduction to deep learning - exercise

## Example

*Human Activity Recognition using Smartphones* dataset

Dataset description:

*The experiments have been carried out with a group of 30 volunteers. Each person performed six activities
(WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) wearing a smartphone.
Using its embedded accelerometer and gyroscope, we captured 3-axial linear acceleration and 3-axial angular velocity.
The experiments have been video-recorded to label the data manually.*

**Variables:**
For each record in the dataset it is provided:
* A 561-feature vector with time and frequency domain variables.
* Its activity label.
* An identifier of the subject who carried out the experiment.

More details at: https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones

### Loading and preparing the data

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim

In [3]:
folder = '../data'  ## put here folder where the file HAR_clean.csv is located

Load the dataset: "HAR_clean.csv"

In [4]:
all_data = pd.read_csv(os.path.join(folder, 'HAR_clean.csv'), index_col=0)

Divide into input and output data

In [5]:
input_data = all_data.iloc[:,:-2].values.astype(np.float32)
input_data.shape

(10299, 561)

In [6]:
output_data = all_data.iloc[:,-1].values
output_data.shape

(10299,)

Divide the data into train and test, keeping 30% for the test

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(input_data, output_data, test_size=0.3)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

(7209, 561) (7209,)
(3090, 561) (3090,)


### Best shallow ML model

In [8]:
from sklearn import svm
from sklearn.model_selection import GridSearchCV

parameters = {'kernel':['linear', 'rbf'], 'C':[1, 10, 100,1000], 'gamma':[0.01, 0.001]}

svm_model_d = svm.SVC()
opt_model_d = GridSearchCV(svm_model_d, parameters)

opt_model_d.fit(X_train, y_train)
print (opt_model_d.best_estimator_)

SVC(C=100, gamma=0.001)


In [9]:
opt_model_d.score(X_test, y_test)

0.9870550161812298

### Training a DL model

In [10]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

le.fit(y_train)

y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

In [11]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train_encoded)

X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test_encoded)

In [12]:
# DNN model - define here the model

class HARNet(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.net(x)


n_features = X_train.shape[1]
n_classes = len(np.unique(y_train))

model = HARNet(n_features, n_classes)

In [13]:
# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [14]:
from sklearn.metrics import accuracy_score

# Training loop
epochs = 30
batch_size = 128
train_losses = []
test_accuracies = []

for epoch in range(epochs):

    # ---- mini-batch training ----
    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], y_train[idx]

        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    train_losses.append(epoch_loss)

    # ---- evaluation ----
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy_score(y_test.numpy(), preds.numpy())
        test_accuracies.append(acc)

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.3f} | Test Acc: {acc:.4f}")

Epoch 01 | Loss: 39.948 | Test Acc: 0.8790
Epoch 02 | Loss: 11.718 | Test Acc: 0.9469
Epoch 03 | Loss: 7.209 | Test Acc: 0.9537
Epoch 04 | Loss: 5.558 | Test Acc: 0.9638
Epoch 05 | Loss: 4.531 | Test Acc: 0.9592
Epoch 06 | Loss: 4.035 | Test Acc: 0.9670
Epoch 07 | Loss: 3.414 | Test Acc: 0.9647
Epoch 08 | Loss: 3.565 | Test Acc: 0.9667
Epoch 09 | Loss: 3.332 | Test Acc: 0.9683
Epoch 10 | Loss: 2.614 | Test Acc: 0.9715
Epoch 11 | Loss: 2.801 | Test Acc: 0.9744
Epoch 12 | Loss: 2.223 | Test Acc: 0.9725
Epoch 13 | Loss: 2.141 | Test Acc: 0.9725
Epoch 14 | Loss: 2.046 | Test Acc: 0.9693
Epoch 15 | Loss: 2.228 | Test Acc: 0.9761
Epoch 16 | Loss: 1.965 | Test Acc: 0.9725
Epoch 17 | Loss: 1.886 | Test Acc: 0.9751
Epoch 18 | Loss: 1.567 | Test Acc: 0.9783
Epoch 19 | Loss: 1.610 | Test Acc: 0.9748
Epoch 20 | Loss: 1.443 | Test Acc: 0.9783
Epoch 21 | Loss: 1.302 | Test Acc: 0.9793
Epoch 22 | Loss: 1.231 | Test Acc: 0.9748
Epoch 23 | Loss: 1.332 | Test Acc: 0.9731
Epoch 24 | Loss: 1.230 | Test Ac

In [15]:
# Test set evaluation
print("\nFinal test accuracy:", test_accuracies[-1])


Final test accuracy: 0.9789644012944984
